In [209]:
import requests
from urllib.parse import urlsplit
from typing import Dict
import requests, urllib.parse


In [211]:
UMLS_KEY = "1d896f23-dbc2-4fd2-bc4b-47ee5a3ca961"
UMLS_REST = "https://uts-ws.nlm.nih.gov/rest"
PUBCHEM   = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def umls_atoms(cui: str, *, version: str = "current") -> list[dict]:
    """
    Low-level helper – returns *all* atom JSON records for a CUI.
    """
    resp = requests.get(
        f"{UMLS_REST}/content/{version}/CUI/{cui}/atoms",
        params={"apiKey": UMLS_KEY},
        timeout=15
    )
    resp.raise_for_status()
    return resp.json()["result"]

def cui_to_rxcui(cui: str) -> str | None:
    """
    UMLS CUI ► RxNorm concept identifier (RxCUI).

    Strategy:
      • iterate over atoms
      • keep only atoms whose rootSource is RXNORM
      • pull the numeric tail of 'sourceConcept' (or 'code')
    """
    for atom in umls_atoms(cui):
        if atom.get("rootSource") == "RXNORM":
            # both 'sourceConcept' and 'code' look like:
            #   'https://uts-ws.nlm.nih.gov/rest/content/2025AA/source/RXNORM/1191'
            url = atom["sourceConcept"]
            return url.rsplit("/", 1)[-1]        # → '1191'
    return None                                  # no RxNorm atom found

In [227]:
def rxcui_to_name(rxcui: str) -> str | None:
    r = requests.get(
        f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json",
        timeout=10
    )
    if r.ok and r.json().get("properties"):
        return r.json()["properties"]["name"]

def pubchem_smiles_from_name(name: str) -> str | None:
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
        f"/compound/name/{requests.utils.quote(name)}/property/CanonicalSMILES/JSON"
    )
    #print(url)
    r = requests.get(url, timeout=10)
    if r.ok:
        return r.json()["PropertyTable"]["Properties"][0]["CanonicalSMILES"]

def chembl_smiles_from_name(name: str, first_only: bool = True) -> str | list[str] | None:
    """
    Resolve `name` in ChEMBL and return the canonical SMILES.
    If `first_only` is False, a list of all hits’ SMILES is returned.
    """
    # 1) hit the Solr-powered search endpoint
    base = "https://www.ebi.ac.uk/chembl/api/data/molecule/search.json"
    q = urllib.parse.quote_plus(name)          # safe URL encoding
    url = f"{base}?q={q}&limit=20"             # grab a few hits in case of ambiguity
    #print(url)                                 # keep your diagnostic print

    r = requests.get(url, timeout=10)
    if not r.ok:
        return None

    molecules = r.json().get("molecules", [])
    if not molecules:
        return None                            # nothing found

    # 2) each hit has `canonical_smiles` at top level; fall back to structure block if needed
    smiles = []
    for mol in molecules:
        smi = (
            mol.get("canonical_smiles")
            or mol.get("molecule_structures", {}).get("canonical_smiles")
        )
        if smi:
            smiles.append(smi)

    if not smiles:
        return None
    return smiles[0] if first_only else smiles

def smiles_from_chembl(chembl_id: str) -> str | None:
    """
    Fetch canonical SMILES from ChEMBL using a ChEMBL compound ID.
    Example ID: 'CHEMBL25'
    """
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}?format=json"
    r = requests.get(url, timeout=10)
    if r.ok:
        mol = r.json()
        smiles = mol.get("molecule_structures", {}).get("canonical_smiles")
        return smiles
    return None

def chembl_id_from_name(name: str) -> str | None:
    """
    Return the ChEMBL compound ID (e.g., 'CHEMBL25') from a drug name.
    """
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={requests.utils.quote(name)}"
    r = requests.get(url, timeout=10)
    if r.ok:
        results = r.json().get("molecules", [])
        if results:
            return results[0].get("molecule_chembl_id")
    return None
    
def atom_priority(atom: Dict) -> int:
    """
    Return a numeric priority for a UMLS atom.
    Lower numbers are better.

    Rules
    -----
    0  RxNorm ingredient terms         (IN or PIN)
    1  Non-RxNorm ingredient terms     (IN or PIN)
    2  RxNorm, but not an ingredient   (BN, SCD, DF, ...)
    3  Everything else                 (all other sources / term types)
    """
    rs = atom.get("rootSource")
    tt = atom.get("termType")

    if rs == "RXNORM" and tt in {"IN", "PIN"}:
        return 0
    if tt in {"IN", "PIN"}:
        return 1
    if rs == "RXNORM":
        return 2
    return 3
    
def cui_to_pubchem_smiles(cui: str) -> str | None:
    print()
    atoms = umls_atoms(cui)                     # from earlier helper
    atoms_eng = [a for a in atoms if a["language"] == "ENG"]
    
    best_atom = min(atoms_eng, key=atom_priority, default=None)
                                 # all other vocabularies
    if not best_atom:
        return None
    print(best_atom)
    name = best_atom["name"]
    chembl_id = chembl_id_from_name(name)
    print(f"looking for {name} with chemblid {chembl_id}")
    return pubchem_smiles_from_name(name)         

In [229]:
cuis = ["C0061633", "C0935916", "C0968383"]   # e.g. aspirin, ibuprofen, paracetamol
smiles_map = {cui: cui_to_pubchem_smiles(cui) for cui in cuis}



{'classType': 'Atom', 'ui': 'A0194332', 'sourceDescriptor': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/source/MSH/C010972', 'sourceConcept': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/source/MSH/M0056908', 'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0061633', 'suppressible': 'false', 'obsolete': 'false', 'rootSource': 'MSH', 'termType': 'CE', 'code': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/source/MSH/C010972', 'language': 'ENG', 'name': 'hydroxyacetaldehyde', 'ancestors': 'NONE', 'descendants': 'NONE', 'attributes': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/AUI/A0194332/attributes', 'relations': 'NONE', 'children': 'NONE', 'parents': 'NONE'}
looking for hydroxyacetaldehyde with chemblid None

{'classType': 'Atom', 'ui': 'A27061467', 'sourceDescriptor': 'NONE', 'sourceConcept': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/source/DRUGBANK/DB00947', 'concept': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0935916', 'suppressible': 'fa

In [231]:
smiles_map

{'C0061633': 'C(C=O)O',
 'C0935916': 'CC12CCC3C(C1CCC2O)C(CC4=C3C=CC(=C4)O)CCCCCCCCCS(=O)CCCC(C(F)(F)F)(F)F',
 'C0968383': 'C1=CC=C2C(=C1)C3=NNC4=CC=CC(=C43)C2=O'}

In [223]:
pubchem_smiles_from_name("SP-600125")

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/SP-600125/property/CanonicalSMILES/JSON


'C1=CC=C2C(=C1)C3=NNC4=CC=CC(=C43)C2=O'

In [221]:
chembl_smiles_from_name("SP-600125")

https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q=SP-600125&limit=20


'Oc1c2ccccc2c2c3c(cccc13)N=N2'

In [203]:
smiles_from_chembl("CHEMBL1725279")

'Oc1c2ccccc2c2c3c(cccc13)N=N2'

In [207]:
from rdkit import Chem

def to_inchikey(smiles: str) -> str | None:
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.inchi.MolToInchiKey(mol)
    except Exception:
        return None

s1 = pubchem_smiles_from_name("Fulvestrant")
s2 = smiles_from_chembl("CHEMBL4760678")

print("InChIKey from PubChem:", to_inchikey(s1))
print("InChIKey from ChEMBL:", to_inchikey(s2))

https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Fulvestrant/property/CanonicalSMILES/JSON
InChIKey from PubChem: VWUXBMIQPBEWFH-UHFFFAOYSA-N
InChIKey from ChEMBL: VWUXBMIQPBEWFH-GXZKXCMSSA-N


### Morgan fingerprints

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# ── Helper functions ──────────────────────────────────────────────────────────
def morgan_features(smiles: str, *, radius: int = 2, n_bits: int = 2048) -> list[int]:
    """
    Return a list of Morgan fingerprint bits for a single SMILES string.

    Falls back to glucose if the SMILES cannot be parsed.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:                                   # simple fallback
        mol = Chem.MolFromSmiles("C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O")

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((1,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.tolist()


# ── Compute fingerprints ───────────────────────────────────────────────────────
fingerprints = [
    morgan_features(smiles) for smiles in senolytics.iloc[:, 1]
]


### physicochemical descriptors

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
from __future__ import annotations

from pathlib import Path
from typing import List

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# ── Descriptor selection ───────────────────────────────────────────────────────
NON_FEATURES = {
    # long exclusion list supplied by the user
    "Chem",
    "PropertyFunctor",
    "_FingerprintDensity",
    "__builtins__",
    "__cached__",
    "__doc__",
    "__file__",
    "__loader__",
    "__name__",
    "__package__",
    "__spec__",
    "_descList",
    "_du",
    "_isCallable",
    "_rdMolDescriptors",
    "_runDoctests",
    "_setupDescriptors",
    "abc",
    "descList",
    "rdMolDescriptors",
    "rdPartialCharges",
    # every AUTOCORR2D_* and other items…
    *[f"AUTOCORR2D_{i}" for i in range(0, 193)],
    "AvgIpc",
    "BCUT2D_CHGHI",
    "BCUT2D_CHGLO",
    "BCUT2D_LOGPHI",
    "BCUT2D_LOGPLOW",
    "BCUT2D_MRHI",
    "BCUT2D_MRLOW",
    "BCUT2D_MWHI",
    "BCUT2D_MWLOW",
    "rdFingerprintGenerator",
    "setupAUTOCorrDescriptors",
    "names",
    "CalcMolDescriptors",
    "_getMorganCountFingerprint",
    "autocorr",
}

# list of (name, function) pairs we *do* want
DESC_FUNCS = [
    (name, func)
    for name, func in Descriptors.descList
    if name not in NON_FEATURES
]

DESC_NAMES = [name for name, _ in DESC_FUNCS]

# ── Helper functions ──────────────────────────────────────────────────────────
FALLBACK_SMILES = "C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O"  # β-D-glucose


def descriptor_vector(smiles: str) -> List[float]:
    """
    Calculate the selected RDKit descriptors for `smiles`.
    Falls back to β-D-glucose if parsing fails.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        mol = Chem.MolFromSmiles(FALLBACK_SMILES)

    return [func(mol) for _, func in DESC_FUNCS]


# ── I/O paths ─────────────────────────────────────────────────────────────────
IN_PATH = Path("smiles_pred.csv")
OUT_PATH = Path("pred_RDKit.csv")


df = pd.read_csv(IN_PATH)

# assume SMILES are in the second column (index 1)
smiles_series = df.iloc[:, 1]

descriptor_rows = [descriptor_vector(smi) for smi in smiles_series]
df_desc = pd.DataFrame(descriptor_rows, columns=DESC_NAMES)

df_out = pd.concat([df, df_desc], axis=1)
df_out.to_csv(OUT_PATH, index=False)

### calculate the similarity between two molecules
- e.g. Tanimoto distances calculated with RDKit physicochemical descriptors to measure the similarity between the drugs